In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
# read the pkl file and load the data
with open('ML_Pipeline/data/data_entire_667_len_613_mean_std.pkl', 'rb') as f:
    data = pickle.load(f)

optimizers = data['optimizers']
sc_r_values_n = data['sc_r_values_n']
sc_eps_values_n = data['sc_eps_values_n']
oc_r_values_n = data['oc_r_values_n']
oc_eps_values_n = data['oc_eps_values_n']
ocx_r_values_n = data['ocx_r_values_n']
ocx_eps_values_n = data['ocx_eps_values_n']
sos_ktheta_values_n = data['sos_ktheta_values_n']
oso_ktheta_values_n = data['oso_ktheta_values_n']
den_results = data['den_results']
se_results = data['se_results']
bm_results = data['bm_results']


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
# read the pkl file and load the data
with open('ML_Pipeline/data/data_entire_667_len_613_mean_std_density_only.pkl', 'rb') as f:
    data = pickle.load(f)

den_results = data['den_results']


In [ ]:
df_full = pd.DataFrame({
    'optimizers': optimizers,
    'sc_r': sc_r_values_n,
    'sc_eps': sc_eps_values_n,
    'oc_r': oc_r_values_n,
    'oc_eps': oc_eps_values_n,
    'ocx_r': ocx_r_values_n,
    'ocx_eps': ocx_eps_values_n,
    'sos_ktheta': sos_ktheta_values_n,
    'oso_ktheta': oso_ktheta_values_n,
    
    'D_11': np.array(den_results['Tob11'])[:,0],
    'D_11H': np.array(den_results['Tob11H'])[:,0],
    'D_14': np.array(den_results['Tob14'])[:,0],
    
    'SE_T11': np.array(se_results['Tob11'])[:,0],
    'SE_T11H': np.array(se_results['Tob11H'])[:,0],
    'SE_T14': np.array(se_results['Tob14'])[:,0],
    
    'BM_T11': np.array(bm_results['Tob11'])[:,0],
    'BM_T11H': np.array(bm_results['Tob11H'])[:,0],
    'BM_T14': np.array(bm_results['Tob14'])[:,0],
    
    'D_11_std': np.array(den_results['Tob11'])[:,1],
    'D_11H_std': np.array(den_results['Tob11H'])[:,1],
    'D_14_std': np.array(den_results['Tob14'])[:,1],
    
    'SE_T11_std': np.array(se_results['Tob11'])[:,1],
    'SE_T11H_std': np.array(se_results['Tob11H'])[:,1],
    'SE_T14_std': np.array(se_results['Tob14'])[:,1],
    
    'BM_T11_std': np.array(bm_results['Tob11'])[:,1],
    'BM_T11H_std': np.array(bm_results['Tob11H'])[:,1],
    'BM_T14_std': np.array(bm_results['Tob14'])[:,1]
    
})
df_full

In [ ]:
df_full.columns

In [ ]:
mean_cols = ['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']
# creat a mask of this 'df_full.iloc[:,1:-3].dropna()'
mask = df_full[mean_cols].dropna().index
df_full_masked = df_full.loc[mask]

In [ ]:
# Define target property values and tolerable errors
target_y = np.array([2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47])

# Define columns for means and stds
mean_cols = ['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']
std_cols = [col + '_std' for col in mean_cols]  # Exclude BM stds for now

# Compute percentage errors (mean and std)
error_mean = 100 * np.abs(df_full_masked[mean_cols].values - target_y[None, :]) / target_y[None, :]
error_std = 100 * df_full_masked[std_cols].values / target_y[None, :]

# Add errors to new DataFrame
df_full_new = df_full_masked.copy()
for i, col in enumerate(mean_cols):
    df_full_new[f'error_{col}'] = error_mean[:, i]
    df_full_new[f'error_{col}_std'] = error_std[:, i]

df_full_new.to_csv(f"ML_Pipeline/data/data_entire_667_len_613_mean_std.csv", index=False)